# B2 exploration · What does exposing a tool actually do?

Start with the [B2 snippets](../snippets/b2/README.md). We will separate the
Python function, advertised schema, requested operation, executing host and
returned observation. Then inspect MCP discovery and failure results.

Default Run All needs no model key or server. Set `RUN_LIVE = True` for actual
agent decisions; enable `RUN_MCP` separately after starting the supplied MCP
server. All records are synthetic. No action sends messages or changes PortOps.

Each experiment gives a prediction and one controlled change. The
[B2 TODO lab](../labs/container_help_desk/B2.md) is the later application exercise.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

TRACK_B = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'snippets/b1').is_dir()), None)
if TRACK_B is None:
    raise RuntimeError('Open this notebook from track-b or track-b/notebooks.')
for env_file in [TRACK_B / 'notebooks/.env', TRACK_B / 'snippets/.env',
                 TRACK_B / '.env', TRACK_B.parent / '.env']:
    load_dotenv(env_file)

RUN_LIVE = False  # Change to True, then restart and Run All for actual model experiments.
MODEL = 'gpt-4.1-mini'
if RUN_LIVE and not os.environ.get('OPEN_AI_KEY'):
    raise ValueError('Configure OPEN_AI_KEY before enabling live calls.')
print('Live inference enabled:', RUN_LIVE)


from agents import Agent, Runner, function_tool, set_default_openai_key, set_tracing_disabled

set_tracing_disabled(True)
if RUN_LIVE:
    set_default_openai_key(os.environ['OPEN_AI_KEY'])

## 1. A function reads; its schema describes

**Predict:** how many reads occur when a callable is wrapped as a tool?
The function body owns the records and the side effect of appending a read log.

In [ ]:
records = {'C1': 'delayed', 'C2': 'on_time'}
reads = []

def get_container_status(container_id: str) -> str:
    """Read a synthetic container status; report unknown IDs without inventing a status."""
    reads.append(container_id)
    return records.get(container_id, 'unknown')

tool = function_tool(get_container_status, failure_error_function=None)
print('Tool name:', tool.name)
print('Description:', tool.description)
print(json.dumps(tool.params_json_schema, indent=2))
print('Reads after describing the tool:', reads)

**Inspect:** the schema names inputs and their constraints. It does not contain
the Python function body or the C1/C2 dictionary. Creating a tool has not read
a container. The model can request this operation only if the application
advertises it; the host still owns its execution.

**Change:** improve the function's docstring, recreate `tool`, and inspect its
description. Recreating the wrapper is necessary after changing a definition.

## 2. A prepared request is not an executed operation

This example is deliberately prepared Python data, not a captured model
request. The host must take another step before any record is read.

In [ ]:
reads.clear()
prepared_request = {'name': tool.name, 'arguments': {'container_id': 'C1'}}
print('Prepared proposal:', prepared_request)
print('Reads before dispatch:', reads)
print('Direct Python observation:', get_container_status(**prepared_request['arguments']))
print('Reads after dispatch:', reads)

**Inspect:** the printed proposal proves no action by itself. The read log is
execution evidence. In a live agent, tool-call items and tool-output items
supply the corresponding distinction.

## 3. Enter the SDK validation boundary without calling a model

To inspect local argument validation, this small supplied adapter passes
prepared JSON through the same `FunctionTool` invocation entry point. It is
**host execution with prepared inputs**, not an autonomous model decision.
`ToolContext` supplies invocation metadata; no provider is contacted.

In [ ]:
from agents.tool_context import ToolContext

async def invoke_prepared(selected_tool, arguments):
    raw = json.dumps(arguments)
    context = ToolContext(context=None, tool_name=selected_tool.name,
                          tool_call_id='prepared-example', tool_arguments=raw)
    return await selected_tool.on_invoke_tool(context, raw)

reads.clear()
print('Observation:', await invoke_prepared(tool, {'container_id': 'C1'}))
print('Executed reads:', reads)

**Predict:** which of the following inputs fails the argument boundary, and
which reaches the function but finds no record? Missing membership is not a
type error. A typed string is not proof that a record exists.

In [ ]:
from agents import ModelBehaviorError

for arguments in [{'container_id': 17}, {}, {'container_id': 'C9'}]:
    reads.clear()
    try:
        observation = await invoke_prepared(tool, arguments)
        print(arguments, '-> observation:', observation)
    except ModelBehaviorError as error:
        print(arguments, '-> argument rejection:', type(error).__name__)
    print('Reads that actually executed:', reads)

**Inspect:** malformed arguments are rejected before the read. C9 is a valid
string, so it reaches the function and returns unknown. Neither case justifies
inventing an observed status. `failure_error_function=None` deliberately keeps
errors visible during this boundary experiment; normal agent applications can
choose a different tool-error policy.

**Change:** try an integer encoded as a string, such as `"17"`. The schema can
accept it even though no such record exists.

## 4. A valid argument can still be outside caller scope

**Predict:** C2 is known and correctly typed. Does that establish permission
to read it? The scope below is application-owned fixture data, not an argument
the model gets to choose. This is a classroom boundary, not authentication.

In [ ]:
allowed_ids = {'C1'}

def get_permitted_status(container_id: str) -> str:
    """Read only a container in this caller's supplied classroom scope."""
    if container_id not in allowed_ids:
        raise PermissionError('Container is outside this caller scope')
    return get_container_status(container_id)

scoped_tool = function_tool(get_permitted_status, failure_error_function=None)
for container_id in ['C1', 'C2']:
    reads.clear()
    try:
        print(container_id, '->', await invoke_prepared(scoped_tool, {'container_id': container_id}))
    except PermissionError as error:
        print(container_id, '->', error)
    print('Underlying reads:', reads)

**Inspect:** C2's denial happens before the underlying read. Schema validity,
record membership and permission answer three different questions.

**Change:** switch the trusted scope to C2 and repeat. Do not add a
model-controlled `allowed_ids` argument to the business tool.

## 5. Compare a real model answer with and without a capability

**Predict:** does an agent name such as “Port assistant” provide record access?
Keep the model, user question and instruction fixed; change only its tools.
A no-tool model might decline or guess. Inspect the actual reply.

In [ ]:
instruction = ('Use recorded statuses only when a tool supplies them. If no source is available, '
               'say so; never invent a status. Unknown means no recorded status.')
without_tools = Agent(name='Port assistant', model=MODEL, instructions=instruction)
with_tools = Agent(name='Port assistant', model=MODEL, instructions=instruction, tools=[tool])
results = {}
if RUN_LIVE:
    for label, agent in [('without tools', without_tools), ('with tools', with_tools)]:
        reads.clear()
        result = await Runner.run(agent, "What is C1's recorded status?", max_turns=4)
        results[label] = result
        print(label, '->', result.final_output)
        print('Actual function reads:', list(reads))
else:
    print('SKIPPED: live model comparison; no prepared model answer is substituted.')

## 6. Open the execution record

**Predict:** is a generated final sentence itself evidence that the function
ran? Inspect distinct request and observation items from the actual run.

In [ ]:
if 'with tools' in results:
    result = results['with tools']
    for item in result.new_items:
        print(item.type, '->', item.raw_item)
    print('Model responses:', len(result.raw_responses))
    print('Tool requests:', sum(item.type == 'tool_call_item' for item in result.new_items))
    print('Tool observations:', sum(item.type == 'tool_call_output_item' for item in result.new_items))
else:
    print('SKIPPED: there is no actual agent run to inspect.')

**Inspect:** the model requests; the host executes; the observation enters the
next decision. A final answer can still misunderstand an observation. Compare
it with the recorded source value, not merely with the count of tool calls.

**Change:** ask for C9, or ask a general shipping-definition question. Does the
chosen execution path fit the information need? A model instruction requests
a behavior; it does not guarantee a particular tool trace.

## 7. Optional · Discovering an MCP tool does not invoke it

In a separate terminal, from `track-b` with the prepared environment active:

```bash
python snippets/b2/04_shared_mcp_server.py
```

Leave that server running. It owns its own synthetic records; changing this
notebook's local dictionary does not change the server's dictionary.
`RUN_MCP` is independent of `RUN_LIVE`: direct MCP inspection needs no model key.

In [ ]:
RUN_MCP = False
MCP_URL = 'http://127.0.0.1:8765/mcp'
remote_catalog = None
if RUN_MCP:
    from agents.mcp import MCPServerStreamableHttp
    async with MCPServerStreamableHttp(name='Classroom status', params={'url': MCP_URL}) as server:
        remote_catalog = await server.list_tools()
        for entry in remote_catalog:
            print(entry.name, entry.description)
        print('Discovery returned descriptions. Inspect the server terminal for actual reads.')
else:
    print('SKIPPED: start the separate MCP server, then set RUN_MCP = True.')

## 8. Unknown record versus tool error over MCP

**Predict:** do unknown C9 and a missing required argument represent the same
outcome? The server can return an ordinary unknown result or mark a tool
invocation as an error. Neither is a usable known status.

In [ ]:
if RUN_MCP:
    async with MCPServerStreamableHttp(name='Classroom status', params={'url': MCP_URL}) as server:
        for arguments in [{'container_id': 'C1'}, {'container_id': 'C9'}, {}]:
            observation = await server.call_tool('get_container_status', arguments)
            print('Arguments:', arguments)
            print('Tool error:', observation.is_error)
            print('Returned content:', observation.content)
else:
    print('SKIPPED: no MCP calls were made.')

**Inspect:** separate a tool-level error result from a connection failure. The
SDK can feed MCP error content back to a model as an observation; successful
HTTP transport alone does not make it a successful business read. Check
`is_error` when inspecting direct results. A stopped server should surface a
connection failure, not silently become a status.

**Change:** add a record in the server snippet, restart that server and invoke
it again. The notebook's client implementation does not need a copy of the new record.

## 9. Optional · Let a real agent use only the remote capability

There are no in-process tools in this agent. The MCP server supplies the
capability while the notebook application still runs the model decision loop.
Both live inference and the separately running MCP server must be enabled.

In [ ]:
if RUN_LIVE and RUN_MCP:
    async with MCPServerStreamableHttp(name='Classroom status', params={'url': MCP_URL}) as server:
        remote_agent = Agent(name='Remote port assistant', model=MODEL,
                             instructions=instruction, mcp_servers=[server])
        remote_result = await Runner.run(remote_agent, "What is C2's recorded status?", max_turns=4)
        print(remote_result.final_output)
else:
    print('SKIPPED: this comparison needs both RUN_LIVE and RUN_MCP.')

## Explain the boundaries

1. What can the model learn from the function schema, and what does it still need a tool to obtain?
2. Which experiment rejected a request before reading any record?
3. How can an unknown ID be schema-valid?
4. Which process owns the records in the MCP experiment?
5. Why are a returned tool error and a network connection failure different?

Apply the mechanisms in the [B2 TODO lab](../labs/container_help_desk/B2.md).
The [LlamaIndex snippet](../snippets/b2/03_llamaindex_agent.py) packages the
same responsibilities with another runtime; framework choice does not change
the source of the record.

Implementation references: [OpenAI tool integration](https://developers.openai.com/api/docs/guides/agents/integrations-observability)
and the installed SDK's `FunctionTool`/`ToolContext` interfaces.